# Fine Tuning BERT for Medical Text Classification
This notebook is a companion of chapter 2 of the "Domain Specific LLMs in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.
The code in this notebook shows how to fine tune a [BERT](https://huggingface.co/bert-base-uncased) model for sequence classification on the [PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA) dataset. The task is to classify whether a PubMed abstract answers a biomedical research question with "yes", "no", or "maybe".
While no hardware acceleration is required to execute all the code cells, it is recommended to use a GPU to speed up the fine tuning step.

## Settings

Install the missing requirements in the Colab VM (HF's Datasets and Accelerate).

In [ ]:
!pip install datasets accelerate scikit-learn

## Data Preparation

Load the expert-labeled subset of the PubMedQA dataset using the HF's Datasets library. This subset contains ~1,000 biomedical question-answer pairs, each labeled as "yes", "no", or "maybe".

In [ ]:
from datasets import load_dataset

dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
print(f"Dataset size: {len(dataset)}")
print(f"Features: {dataset.features}")

Display a sample to understand the dataset structure:

In [ ]:
dataset[0]

The important fields are:

- `question`: a biomedical research question.
- `context`: the relevant sentences from the PubMed abstract (provided as `contexts`, a list of strings).
- `long_answer`: the full conclusion from the abstract.
- `final_decision`: the label — "yes", "no", or "maybe".

We will concatenate the question and the long answer as input, and use `final_decision` as the classification target.

Map the string labels to integer IDs:

In [ ]:
from datasets import ClassLabel

label2id = {"yes": 0, "no": 1, "maybe": 2}
id2label = {v: k for k, v in label2id.items()}

dataset = dataset.map(lambda x: {"label": label2id[x["final_decision"]]})
dataset = dataset.cast_column("label", ClassLabel(names=["yes", "no", "maybe"]))

print("Label distribution:")
for label_name, label_id in label2id.items():
    count = sum(1 for x in dataset if x["label"] == label_id)
    print(f"  {label_name}: {count}")

Split the dataset into train and test sets (80%/20%):

In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

Load a BERT tokenizer to process the question and long answer fields:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Define a preprocessing function that tokenizes the question and long answer as a sentence pair. This lets BERT leverage its next-sentence-prediction pre-training to understand the relationship between question and answer.

In [ ]:
def preprocess_function(examples):
    return tokenizer(
        examples["question"],
        examples["long_answer"],
        max_length=512,
        truncation=True,
        padding="max_length",
    )

Apply the preprocessing function over the entire dataset. By setting `batched=True`, multiple elements of the dataset will be processed at once.

In [ ]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=[c for c in dataset["train"].column_names if c != "label"],
)

## Evaluation Metric

Define a function to compute accuracy and macro F1 score during training:

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }

## Fine Tuning

Load BERT with *AutoModelForSequenceClassification*, configured for 3 output classes (yes, no, maybe):

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    device_map="auto",
)

Define the training hyperparameters in a *TrainingArguments* instance:

In [ ]:
training_args = TrainingArguments(
    output_dir="bert_pubmedqa_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    push_to_hub=False,
    report_to="none",
)

Pass the training arguments to a *Trainer* instance along with the model, dataset, tokenizer, and metrics function:

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Start the fine tuning:

In [ ]:
trainer.train()

## Evaluation

Evaluate the fine-tuned model on the test set:

In [ ]:
results = trainer.evaluate()
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"Macro F1: {results['eval_f1_macro']:.4f}")

## Inference

The fine-tuned model can now be used for inference. Let's provide a biomedical question and an abstract conclusion to classify:

In [ ]:
question = "Do statins reduce the risk of cardiovascular events in patients with type 2 diabetes?"
answer = "Statin therapy significantly reduced the incidence of major cardiovascular events in diabetic patients. The relative risk reduction was consistent across all subgroups analyzed, supporting the use of statins in this population."

Tokenize the text and return PyTorch tensors:

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

inputs = tokenizer(question, answer, return_tensors="pt", truncation=True, max_length=512)
inputs = inputs.to(device)

Pass the inputs to the model and get the predicted class:

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)
predicted_class = torch.argmax(probabilities, dim=-1).item()

print(f"Predicted answer: {id2label[predicted_class]}")
print(f"Confidence scores:")
for label_name, prob in zip(label2id.keys(), probabilities[0].tolist()):
    print(f"  {label_name}: {prob:.4f}")